# 04. Afinado de un transformer

**MAP - Charting Student Math Misunderstandings**

Este cuaderno **requiere GPU**. Antes de ejecutarlo, en Colab: `Entorno de ejecución` y luego
`Cambiar tipo de entorno de ejecución`, y elegir **T4 GPU**.

El propósito es contrastar la representación densa y contextual de un transformer contra las
representaciones de frecuencia del cuaderno anterior. Interesa especialmente comprobar si la
dificultad reportada por los organizadores de la competencia, que atribuyen el fracaso de los
modelos preentrenados a la complejidad del contenido matemático, se reproduce aquí.

## 0. Arranque

Este cuaderno funciona igual en **Google Colab** y en una instalación local.

En Colab la celda siguiente clona el repositorio, instala las dependencias y deja el proyecto
listo. Para que la descarga de los datos funcione hay que registrar antes el token de Kaggle
como secreto: en el panel izquierdo de Colab, icono de la llave, agregar un secreto llamado
`KAGGLE_ACCESS_TOKEN` con el contenido de `~/.kaggle/access_token` y habilitar el acceso desde
el cuaderno.

En local la celda no hace nada salvo ubicar la raíz del proyecto.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPOSITORIO = "https://github.com/Tunchxz/math-misconception-map.git"
CARPETA = "math-misconception-map"

if EN_COLAB:
    if not Path(CARPETA).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPOSITORIO, CARPETA], check=True)
    os.chdir(f"/content/{CARPETA}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
    )
    try:
        from google.colab import userdata

        os.environ["KAGGLE_ACCESS_TOKEN"] = userdata.get("KAGGLE_ACCESS_TOKEN")
        print("Token de Kaggle cargado desde los secretos de Colab.")
    except Exception:
        print("Registre el secreto KAGGLE_ACCESS_TOKEN para poder descargar los datos.")

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("Entorno:", "Google Colab" if EN_COLAB else "local")
print("Raíz del proyecto:", RAIZ)

## 1. Preparación del entorno

La primera celda detecta si el cuaderno corre en Colab y, en ese caso, instala las dependencias
y descarga los datos. En local no hace nada, de modo que el cuaderno funciona en ambos sitios.

In [ ]:
# El afinado requiere bibliotecas que no forman parte de requirements.txt,
# porque solo se usan en este cuaderno.
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.40,<5", "datasets>=2.19", "accelerate>=0.30"],
    check=True,
)

import numpy as np
import pandas as pd
import torch

print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))
    print("Memoria:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("ATENCIÓN: sin GPU el afinado tardará horas.")
    print("En Colab: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU")

## 2. Obtención de los datos

En Colab se descarga el conjunto limpio desde el repositorio, o bien se sube manualmente el
archivo `train_limpio.parquet` que produjo el cuaderno 01. En local se lee de `data/processed`.

In [ ]:
from src.data.make_dataset import asegurar_datos

train = asegurar_datos()
print("Observaciones:", f"{len(train):,}")
train.head(3)

In [ ]:
SEMILLA = 42
SIN_MISCONCEPTION = "NA"
ALIAS = {"Wrong_Fraction": "Wrong_fraction"}

train["misconception_unificada"] = train["misconception"].map(lambda v: ALIAS.get(v, v))
train["objetivo_unificado"] = train["Category"] + ":" + train["misconception_unificada"]

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

print("Clases de Category:", train["Category"].nunique())
print("Errores conceptuales:", train["misconception_unificada"].nunique() - 1)

## 3. Las dos formas de entrada

Se comparan dos maneras de presentar el texto al modelo.

- **Solo la explicación.** El modelo ve únicamente lo que escribió el estudiante.
- **Explicación con contexto.** Se antepone la respuesta seleccionada, separada por el token
  especial del modelo.

La comparación importa porque el análisis exploratorio mostró que el contexto de la pregunta
está casi perfectamente asociado con la etiqueta. Dar demasiado contexto puede inducir al modelo
a memorizar en lugar de razonar sobre el texto.

In [ ]:
def construir_entradas(df, con_contexto, separador):
    '''Devuelve la lista de textos de entrada del modelo.'''
    explicacion = df["explicacion_limpia"].astype(str)
    if not con_contexto:
        return explicacion.tolist()
    return (df["MC_Answer_limpia"].astype(str) + f" {separador} " + explicacion).tolist()


print("Sin contexto:", construir_entradas(train.head(2), False, "[SEP]"))
print("Con contexto:", construir_entradas(train.head(2), True, "[SEP]"))

## 4. Partición por pregunta

Se usa el esquema agrupado, que deja preguntas completas fuera del entrenamiento. Es el más
exigente y el único que refleja lo que la competencia evalúa, de modo que el resultado es
comparable con el del cuaderno anterior.

In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
idx_tr, idx_te = next(iter(gkf.split(train, train["Category"], train["QuestionId"])))

entrenamiento = train.iloc[idx_tr].reset_index(drop=True)
prueba = train.iloc[idx_te].reset_index(drop=True)

print(f"Entrenamiento: {len(entrenamiento):,} filas, "
      f"{entrenamiento['QuestionId'].nunique()} preguntas")
print(f"Prueba:        {len(prueba):,} filas, {prueba['QuestionId'].nunique()} preguntas")
print("Preguntas compartidas:",
      len(set(entrenamiento['QuestionId']) & set(prueba['QuestionId'])))

## 5. Afinado del modelo

Se emplea **MiniLM-L12**, una versión destilada de BERT con unos 33 millones de parámetros.
Wang et al. (2020) la propusieron comprimiendo las relaciones de autoatención del modelo
original. Su tamaño reducido permite entrenarla en los recursos gratuitos de Colab y, sobre
todo, cargarla dentro de la aplicación interactiva del proyecto.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

MODELO_BASE = "microsoft/MiniLM-L12-H384-uncased"
LONGITUD_MAX = 128
EPOCAS = 3
LOTE = 32

tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)
print("Token separador:", tokenizador.sep_token)
print("Vocabulario:", tokenizador.vocab_size)

In [ ]:
def preparar_dataset(df, codificador, con_contexto):
    '''Tokeniza los textos y adjunta las etiquetas codificadas.'''
    textos = construir_entradas(df, con_contexto, tokenizador.sep_token)
    ds = Dataset.from_dict(
        {"text": textos, "labels": codificador.transform(df["Category"])}
    )
    return ds.map(
        lambda lote: tokenizador(lote["text"], truncation=True, max_length=LONGITUD_MAX),
        batched=True,
        remove_columns=["text"],
    )


def metricas(pred):
    '''Calcula F1 macro y ponderado a partir de la salida del Trainer.'''
    etiquetas = pred.label_ids
    predichas = pred.predictions.argmax(-1)
    return {
        "f1_macro": f1_score(etiquetas, predichas, average="macro", zero_division=0),
        "f1_ponderado": f1_score(etiquetas, predichas, average="weighted", zero_division=0),
    }

In [ ]:
def afinar(con_contexto, etiqueta_experimento):
    '''Afina el modelo con la forma de entrada indicada y devuelve sus métricas.'''
    codificador = LabelEncoder().fit(train["Category"])
    ds_tr = preparar_dataset(entrenamiento, codificador, con_contexto)
    ds_te = preparar_dataset(prueba, codificador, con_contexto)

    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODELO_BASE, num_labels=len(codificador.classes_)
    )

    args = TrainingArguments(
        output_dir=f"salida_{etiqueta_experimento}",
        num_train_epochs=EPOCAS,
        per_device_train_batch_size=LOTE,
        per_device_eval_batch_size=LOTE * 2,
        learning_rate=3e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        seed=SEMILLA,
        fp16=torch.cuda.is_available(),
        report_to=[],
    )

    entrenador = Trainer(
        model=modelo,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_te,
        data_collator=DataCollatorWithPadding(tokenizador),
        compute_metrics=metricas,
    )

    import time
    t0 = time.perf_counter()
    entrenador.train()
    segundos = time.perf_counter() - t0

    evaluacion = entrenador.evaluate()
    return {
        "experimento": etiqueta_experimento,
        "f1_macro": evaluacion["eval_f1_macro"],
        "f1_ponderado": evaluacion["eval_f1_ponderado"],
        "segundos": segundos,
    }, entrenador, codificador

In [ ]:
resultado_sin, entrenador_sin, codificador = afinar(False, "solo_explicacion")
print(resultado_sin)

In [ ]:
resultado_con, entrenador_con, _ = afinar(True, "con_contexto")
print(resultado_con)

## 6. Comparación de las dos formas de entrada

In [ ]:
comparacion = pd.DataFrame([resultado_sin, resultado_con])
comparacion["f1_macro"] = comparacion["f1_macro"].round(4)
comparacion["f1_ponderado"] = comparacion["f1_ponderado"].round(4)
comparacion["minutos"] = (comparacion["segundos"] / 60).round(1)
comparacion[["experimento", "f1_macro", "f1_ponderado", "minutos"]]

## 7. Comparación con los modelos clásicos

Se contrastan estos resultados con los del cuaderno 03, bajo el mismo esquema de validación
agrupada por pregunta y sobre la misma etapa.

In [ ]:
from src.data.make_dataset import cargar_transformado

try:
    clasicos = cargar_transformado("metricas_clasicos")
except FileNotFoundError:
    print("Ejecute antes el cuaderno 03 para poder comparar.")
    clasicos = pd.DataFrame()

## 8. Guardado del modelo

Se guarda la mejor de las dos variantes, en el formato que la aplicación puede cargar.

In [ ]:
mejor = entrenador_con if resultado_con["f1_macro"] >= resultado_sin["f1_macro"] else entrenador_sin
etiqueta = (
    "con_contexto" if resultado_con["f1_macro"] >= resultado_sin["f1_macro"]
    else "solo_explicacion"
)

destino = Path("modelo_minilm")
mejor.save_model(destino)
tokenizador.save_pretrained(destino)
pd.Series(codificador.classes_).to_csv(destino / "clases.csv", index=False, header=False)
comparacion.to_csv(destino / "metricas.csv", index=False)

print("Variante guardada:", etiqueta)
print("Archivos:", sorted(p.name for p in destino.iterdir()))
tamano = sum(p.stat().st_size for p in destino.rglob("*")) / 1e6
print(f"Tamaño total: {tamano:.1f} MB")

if "google.colab" in sys.modules:
    import shutil

    shutil.make_archive("modelo_minilm", "zip", destino)
    from google.colab import files

    files.download("modelo_minilm.zip")
    print("Descomprima el archivo dentro de models/ del repositorio local.")